# Graph Neural Network for Network Intrusion Detection
## Mid-Semester Progress Demo

**Master's Level Project**  
**Dataset:** UNSW-NB15  
**Model:** GraphSAGE (Graph Sample and Aggregate)

---
## 1. Project Overview

### Objective
Develop a Graph Neural Network-based intrusion detection system that models network traffic flows as graphs to capture complex relationships between network entities and detect malicious activities.

### Key Innovation
Traditional IDS approaches treat network flows independently. Our GNN approach:
- **Models relationships** between flows through graph structure
- **Captures network topology** patterns that indicate coordinated attacks
- **Leverages message passing** to aggregate information from neighboring flows
- **Learns hierarchical features** through multi-layer graph convolutions

### Current Status
✅ Data preprocessing pipeline complete  
✅ Graph construction from network flows  
✅ GraphSAGE model trained and saved (`graphsage_best.pt`)  
✅ Evaluation metrics documented  
✅ React-based frontend demo UI  
⚠️ Backend integration pending (frontend shows sample data)

---
## 2. Technology Stack

### Backend & Machine Learning
| Technology | Purpose | Version |
|------------|---------|--------|
| **Python** | Core language | 3.8+ |
| **PyTorch** | Deep learning framework | Latest |
| **PyTorch Geometric** | Graph neural networks | Latest |
| **Pandas** | Data manipulation | Latest |
| **NumPy** | Numerical computing | Latest |
| **Scikit-learn** | Preprocessing & baselines | Latest |
| **CUDA** | GPU acceleration | Optional |

### Frontend & Visualization
| Technology | Purpose |
|------------|--------|
| **React** | User interface framework |
| **Recharts** | Performance metrics visualization |
| **React Flow** | Graph topology visualization |
| **Tailwind CSS** | Styling |

### Testing & Development
- **pytest**: Unit testing for preprocessing and inference
- **Git**: Version control

---
## 3. Dataset: UNSW-NB15

### Dataset Overview
The UNSW-NB15 dataset is a comprehensive network intrusion detection dataset created by the Cyber Range Lab at UNSW Canberra. It contains modern normal activities and synthetic attack behaviors.

### Dataset Statistics

#### Training Set
- **Total flows:** 175,341
- **Normal flows:** 56,000 (31.9%)
- **Attack flows:** 119,341 (68.1%)

#### Test Set
- **Total flows:** 82,332
- **Normal flows:** 37,000 (44.9%)
- **Attack flows:** 45,332 (55.1%)

#### Attack Categories
The dataset includes 9 attack types:
1. **Fuzzers** - Attempts to cause program/network suspension
2. **Analysis** - Port scans, spam
3. **Backdoors** - Bypassing authentication mechanisms
4. **DoS** - Denial of Service attacks
5. **Exploits** - Exploiting vulnerabilities
6. **Generic** - Hash collision attacks
7. **Reconnaissance** - Information gathering
8. **Shellcode** - Exploiting vulnerabilities with shellcode
9. **Worms** - Self-replicating malware

### Feature Space
- **Original columns:** 45
- **After preprocessing:** 42 features
  - 39 numerical features (scaled with StandardScaler)
  - 3 categorical features (encoded with OrdinalEncoder)
- **Target variables:** 
  - `label`: Binary (0=Normal, 1=Attack)
  - `attack_cat`: Multi-class attack category

### Key Features Include:
- **Flow-based:** Duration, protocol, service, state
- **Content-based:** Payload size, TCP flags, header lengths
- **Time-based:** Inter-arrival times, jitter
- **Connection-based:** Source/destination IPs and ports

---
## 4. Data Preprocessing Pipeline

### Pipeline Overview
Our preprocessing pipeline (`src/data_preprocessing.py`) transforms raw network flow data into GNN-ready graph structures.

### Processing Steps

#### Step 1: Data Loading & Cleaning
```python
# Load CSV files
train_df = pd.read_csv('UNSW_NB15_training-set.csv')
test_df = pd.read_csv('UNSW_NB15_testing-set.csv')

# Drop identifier column (not a feature)
train_df = train_df.drop('id', axis=1)
test_df = test_df.drop('id', axis=1)
```

#### Step 2: Feature Engineering
- **Numerical features:** 39 features scaled using StandardScaler
  - Ensures zero mean and unit variance
  - Prevents features with large magnitudes from dominating
- **Categorical features:** 3 features encoded using OrdinalEncoder
  - `proto`: Protocol type
  - `service`: Network service
  - `state`: Connection state

#### Step 3: Graph Construction
Each network flow window becomes a graph:
- **Nodes:** Individual network flows
- **Node features:** 42-dimensional feature vectors
- **Edges:** Connections between related flows based on:
  - Shared source/destination IPs
  - Similar timestamps (temporal proximity)
  - Protocol relationships
- **Labels:** Binary attack/normal classification

#### Step 4: Serialization
Processed graphs saved as PyTorch tensors:
- **Training graphs:** `data/processed/graphs/train/graph_*.pt`
- **Test graphs:** `data/processed/graphs/test/graph_*.pt`
- **Statistics:** `overall_stats.json`, `*_graph_stats.csv`

### Preprocessing Artifacts
```
data/processed/graphs/
├── train/           # 1,754 training graphs
├── test/            # 824 test graphs
├── overall_stats.json
└── test_graph_stats.csv
```

---
## 5. GraphSAGE Architecture

### Why GraphSAGE?
GraphSAGE (Graph Sample and Aggregate) is particularly suited for intrusion detection because:
- **Inductive learning**: Can generalize to unseen graph structures (new attack patterns)
- **Scalability**: Samples fixed-size neighborhoods rather than using full graph
- **Aggregation**: Combines information from neighboring nodes effectively

### Model Architecture

```
Input: Graph G = (V, E, X)
  V: nodes (network flows)
  E: edges (flow relationships)
  X: node features (42-dim vectors)

Layer 1: SAGEConv(42 → 64)
  ├─ Sample neighbors
  ├─ Aggregate neighbor features (mean)
  ├─ Concatenate with self features
  └─ Linear transformation + ReLU

Layer 2: SAGEConv(64 → 32)
  ├─ Sample 2-hop neighbors
  ├─ Aggregate features
  └─ Linear transformation + ReLU

Output Layer: Linear(32 → 2)
  └─ Binary classification (Normal/Attack)
```

### Key Hyperparameters
- **Hidden dimensions:** 64 → 32
- **Number of layers:** 2 (to capture 2-hop neighborhood)
- **Aggregation:** Mean aggregation
- **Activation:** ReLU
- **Output:** 2 classes (binary classification)

### Message Passing Formula
For each node $v$:
```
h_v^{(k)} = σ(W^{(k)} · CONCAT(h_v^{(k-1)}, AGG({h_u^{(k-1)} : u ∈ N(v)})))
```
Where:
- $h_v^{(k)}$: node embedding at layer k
- $N(v)$: sampled neighbors of v
- $AGG$: mean aggregation
- $W^{(k)}$: learnable weight matrix
- $σ$: ReLU activation

---
## 6. Training Process

### Training Configuration
```python
optimizer = Adam(lr=0.001)
criterion = CrossEntropyLoss()
epochs = 50  # with early stopping
batch_size = 32
device = 'cuda' if available else 'cpu'
```

### Training Loop
1. **Forward Pass**: Compute predictions for batch of graphs
2. **Loss Computation**: Cross-entropy loss
3. **Backward Pass**: Gradient computation
4. **Optimization**: Adam optimizer update
5. **Validation**: Evaluate on test set every epoch
6. **Early Stopping**: Save best model, stop if no improvement

### Model Artifacts
- **Saved model:** `models/graphsage_best.pt`
- **Contains:**
  - Model state dict (learned weights)
  - Optimizer state
  - Training metadata (epoch, loss)

### Training Infrastructure
- **Module:** `src/model.py` (GraphSAGE class)
- **Training script:** `src/train.py`
- **Data loaders:** PyTorch Geometric DataLoader
- **Hardware:** GPU-accelerated (CUDA) when available

---
## 7. Model Performance

### Evaluation Metrics (on Test Set)

| Metric | Score |
|--------|-------|
| **Accuracy** | 88.17% |
| **Precision** | 84.59% |
| **Recall** | 96.01% |
| **F1-Score** | 89.94% |

### Performance Analysis

#### Strengths
- **High Recall (96.01%)**: Excellent at detecting attacks - only 4% of attacks missed
- **Strong F1-Score (89.94%)**: Good balance between precision and recall
- **Solid Accuracy (88.17%)**: Reliable overall classification

#### Tradeoffs
- **Precision (84.59%)**: ~15% false positive rate
  - Some normal traffic flagged as malicious
  - Acceptable tradeoff for security (better safe than sorry)

### Comparison with Traditional ML
GraphSAGE advantages over traditional models (e.g., Random Forest):
- **Relational learning**: Captures attack patterns spanning multiple flows
- **Topology awareness**: Detects coordinated attacks through graph structure
- **Feature interaction**: Learns complex non-linear relationships

### Evaluation Module
- **Script:** `src/evaluate.py`
- **Outputs:** Classification report, confusion matrix
- **Metrics:** Sklearn classification metrics

---
## 8. Inference Pipeline

### Real-time Prediction Flow
```python
# src/inference.py
predictor = GNNPredictor(
    model_path='models/graphsage_best.pt',
    preprocessor_path='models/preprocessor.pkl'
)

# Load and predict on new CSV
results = predictor.predict_csv('new_traffic.csv')
```

### Prediction Steps
1. **Load CSV**: Read network flow records
2. **Preprocess**: Apply saved StandardScaler and OrdinalEncoder
3. **Graph Construction**: Build graph from flows
4. **Model Inference**: Forward pass through GraphSAGE
5. **Postprocess**: Extract predictions and confidence scores

### Output Format
```json
{
  "predictions": [0, 1, 0, 1, ...],
  "probabilities": [[0.92, 0.08], [0.15, 0.85], ...],
  "labels": ["Normal", "Attack", "Normal", "Attack", ...]
}
```

---
## 9. Frontend Demo Application

### UI Components
A React-based dashboard for visualizing intrusion detection results:

#### 1. Upload Section (`frontend/src/components/UploadSection.jsx`)
- CSV file upload interface
- **Current Status:** Accepts files but shows sample data
- **Note:** Backend integration pending

#### 2. Results Dashboard (`frontend/src/components/ResultsDashboard.jsx`)
- Summary metrics: total flows, attack rate
- Visual indicators for system health

#### 3. Flow Table (`frontend/src/components/FlowTable.jsx`)
- Detailed flow-level predictions
- Displays: Flow ID, Source/Dest IPs, Prediction, Confidence

#### 4. Graph Visualization (`frontend/src/components/GraphVisualization.jsx`)
- Interactive network topology view
- Node coloring: Red (attack), Green (normal)
- Built with React Flow

### Sample Data Display
**Current frontend behavior:**
- Shows 250 sample flow predictions
- Displays hardcoded metrics: 25.2% attack rate
- 20-node sample graph visualization
- All data from `frontend/src/data/sampleData.js`

### Next Steps for Integration
- [ ] Create Flask API endpoint (`/api/predict`)
- [ ] Connect upload handler to Python inference pipeline
- [ ] Stream real predictions to frontend
- [ ] Real-time graph updates from model output

---
## 10. Project Structure

```
gnn_intrusion_detection/
├── data/
│   ├── raw/                    # Original UNSW-NB15 CSV files
│   └── processed/
│       └── graphs/             # Preprocessed graph tensors
│           ├── train/          # 1,754 training graphs
│           ├── test/           # 824 test graphs
│           └── overall_stats.json
├── src/
│   ├── data_preprocessing.py   # Feature engineering + graph construction
│   ├── model.py                # GraphSAGE model definition
│   ├── train.py                # Training loop
│   ├── evaluate.py             # Performance metrics
│   └── inference.py            # Real-time prediction pipeline
├── models/
│   ├── graphsage_best.pt       # Trained model weights
│   └── preprocessor.pkl        # Saved scalers/encoders
├── frontend/
│   ├── src/
│   │   ├── components/         # React UI components
│   │   └── data/sampleData.js  # Demo data
│   └── package.json
├── notebooks/
│   └── project_progress_demo.ipynb  # This notebook
└── tests/
    └── test_*.py               # Pytest unit tests
```

---
## 11. Key Achievements

### Completed Milestones ✅
1. **Data Engineering**
   - Processed 257,673 network flows
   - Built 2,578 graph structures (1,754 train + 824 test)
   - Engineered 42-dimensional feature space

2. **Model Development**
   - Implemented GraphSAGE architecture from scratch
   - Trained model with 96.01% recall
   - Achieved 89.94% F1-score on test set

3. **Infrastructure**
   - GPU-accelerated training pipeline
   - Reusable inference module
   - Comprehensive unit tests

4. **Visualization**
   - Interactive React dashboard
   - Graph topology viewer
   - Real-time metrics display (demo)

### Current Limitations ⚠️
- Frontend uses sample data (backend API not connected)
- No live streaming of network traffic
- Single model (no ensemble)

### Future Enhancements 🚀
1. **Backend Integration**
   - Flask/FastAPI endpoint for predictions
   - WebSocket for real-time updates

2. **Model Improvements**
   - Attention mechanisms (GAT)
   - Multi-class attack classification
   - Explainability (GNNExplainer)

3. **Production Readiness**
   - Model versioning
   - A/B testing framework
   - Performance monitoring

---
## 12. Technical Insights

### Why Graph Neural Networks for IDS?

**Traditional ML approaches treat flows independently:**
```
Flow 1 → [Features] → Model → Prediction 1
Flow 2 → [Features] → Model → Prediction 2
Flow 3 → [Features] → Model → Prediction 3
```
❌ Misses relationships between flows  
❌ Cannot detect coordinated attacks  
❌ Ignores network topology

**GNN approach models the network structure:**
```
    Flow 1 ←→ Flow 2
      ↓  ⤢    ↓  ⤢
    Flow 3 ←→ Flow 4
         ↘   ↙
        Model
```
✅ Captures flow relationships  
✅ Detects multi-stage attacks  
✅ Leverages graph topology

### Real-World Attack Scenarios GNNs Excel At:
1. **Port Scanning**: Multiple flows from one source → many destinations
2. **DDoS**: Many sources → one destination in short time window
3. **Lateral Movement**: Sequential connections through network
4. **Botnet Communication**: Coordinated traffic patterns

### Graph Construction Strategy
**Edge creation heuristics:**
- Same source IP → temporal edge
- Same destination IP → target edge
- Same subnet → network edge
- Protocol similarity → behavioral edge

This creates a rich graph structure that GraphSAGE can learn from.

---
## 13. Demo Instructions

### Running the Frontend Demo

```bash
# Navigate to frontend directory
cd frontend

# Install dependencies (first time only)
npm install

# Start development server
npm run dev

# Open browser to http://localhost:5173
```

### Demo Features to Showcase
1. **Upload Interface**: Drag-and-drop CSV file area
2. **Summary Dashboard**: Attack rate, total flows, system health
3. **Flow Table**: Sortable, filterable prediction table
4. **Graph Visualization**: Interactive network topology
   - Zoom/pan controls
   - Node coloring by prediction
   - Hover for details

### Sample Data Currently Displayed
- **Total Flows**: 250
- **Attack Rate**: 25.2% (63 malicious flows)
- **Graph**: 20 nodes, 45 edges
- **Predictions**: Random confidence scores (0.1 - 0.99)

**Note**: Upload feature accepts CSV files but displays sample data until backend is integrated.

### Python Inference (Backend Ready)

```python
# Test the inference pipeline
from src.inference import GNNPredictor

predictor = GNNPredictor(
    model_path='models/graphsage_best.pt',
    preprocessor_path='models/preprocessor.pkl'
)

# Predict on CSV
results = predictor.predict_csv('test_data.csv')
print(f"Detected {sum(results['predictions'])} attacks")
```

---
## 14. Questions & Discussion

### Potential Discussion Points

1. **Model Architecture**
   - Why 2 layers? Would deeper help?
   - Alternative aggregation functions (max, LSTM)?

2. **Graph Construction**
   - How to handle dynamic/streaming graphs?
   - Optimal edge thresholds for IP/time similarity?

3. **Performance**
   - High recall vs. precision tradeoff acceptable?
   - How to reduce false positives (84% precision)?

4. **Scalability**
   - Real-time processing latency?
   - Handling high-volume traffic (10k+ flows/sec)?

5. **Explainability**
   - Which graph features contribute most to attack detection?
   - Can we visualize attention/importance scores?

### Future Work Priorities
- [ ] Backend API integration
- [ ] Multi-class attack type classification
- [ ] Graph attention networks (GAT)
- [ ] Real-time streaming inference
- [ ] Model explainability (GNNExplainer)
- [ ] Benchmark against commercial IDS

---
## 15. References

### Dataset
- Moustafa, N., & Slay, J. (2015). UNSW-NB15: a comprehensive data set for network intrusion detection systems. *Military Communications and Information Systems Conference (MilCIS)*.

### GraphSAGE
- Hamilton, W., Ying, Z., & Leskovec, J. (2017). Inductive Representation Learning on Large Graphs. *NeurIPS 2017*.

### PyTorch Geometric
- Fey, M., & Lenssen, J. E. (2019). Fast Graph Representation Learning with PyTorch Geometric. *ICLR 2019 Workshop on Representation Learning on Graphs and Manifolds*.

### GNN for Intrusion Detection
- Lo, W. W., Layeghy, S., Sarhan, M., Gallagher, M., & Portmann, M. (2022). E-GraphSAGE: A Graph Neural Network based Intrusion Detection System. *arXiv preprint arXiv:2103.16329*.

---

## Thank You!

**Questions?**

---

*This notebook was created for mid-semester project demonstration.*